# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

Y. Wu  ->  Y. Wu  |  ['Y. Wu']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
J. Matharu  ->  J. Matharu  |  ['J. Matharu']
M. McCaughrean  ->  M. McCaughrean  |  ['M. McCaughrean']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
S. Saha  ->  S. Saha  |  ['S. Saha']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
M. Benisty  ->  M. Benisty  |  ['M. Benisty']
M. Flock  ->  M. Flock  |  ['M. Flock']
Y. Wu  ->  Y. Wu  |  ['Y. Wu']
Y. Wang  ->  Y. Wang  |  ['Y. Wang']


Y. Wang  ->  Y. Wang  |  ['Y. Wang']
S. Kraus  ->  S. Kraus  |  ['S. Kraus']
Arxiv has 124 new papers today
          11 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/11 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.00195


extracting tarball to tmp_2611.00195... done.
Retrieving document from  https://arxiv.org/e-print/2611.00202


extracting tarball to tmp_2611.00202... done.
Retrieving document from  https://arxiv.org/e-print/2611.00215


extracting tarball to tmp_2611.00215...

 done.


... (13 outputs left out)

 done.
Retrieving document from  https://arxiv.org/e-print/2611.02397
extracting tarball to tmp_2611.02397...

 done.
Retrieving document from  https://arxiv.org/e-print/2611.02934


extracting tarball to tmp_2611.02934...

 done.


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.00215-b31b1b.svg)](https://arxiv.org/abs/2611.00215) | **A VLT/CRIRES+ study of helium 1.083$μ$m absorption features towards the inner Orion Nebula**  |
|| T. J. Haworth, et al. -- incl., <mark>M. McCaughrean</mark> |
|*Appeared on*| *2026-11-03*|
|*Comments*| *Accepted for publication in MNRAS*|
|**Abstract**|            JWST NIRCam F115W observations towards the inner Orion Nebula revealed a diverse array of previously unknown dark features. The most likely cause was absorption or suppressed emission of the helium 1.083$\mu$m triplet which lies in that filter. We have obtained VLT/CRIRES+ high-resolution spectrographic observations over a selection of different dark targets, confirming that the features are indeed explained by a reduction in the flux of the helium 1.083$\mu$m triplet, a reduction that is fairly uniform across the line. The Orion Nebula is a blister HII region, with a dense, bright ionised layer on the far side from the observer near the Orion A molecular cloud and lower density gas elsewhere. Using a simple radiative transfer model, we show that dark features arise when emission from the dense layer is absorbed by intermediate density helium (e.g. in clumps, globules, or winds from objects like proplyds) on the near side of the dense layer.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.01209-b31b1b.svg)](https://arxiv.org/abs/2611.01209) | **exoALMA XXV: A Statistical Analysis of Gaseous Substructure in Low-Inclination Disks**  |
|| R. Teague, et al. -- incl., <mark>M. Benisty</mark>, <mark>M. Flock</mark> |
|*Appeared on*| *2026-11-03*|
|*Comments*| *Accepted to exoALMA ApJL Focus Issue*|
|**Abstract**|            Protoplanetary disks exhibit complex structure in both their gas dynamics and their temperature. Characterizing it has proven challenging, owing to the mismatch between the observed complexity and the relatively simple semi-analytical models typically used to interpret it. To augment these analyses, we apply the second-order structure function to the velocity and temperature perturbations of four low-inclination sources from the exoALMA program, each measured against a noise-only null built from the data themselves. All exhibit correlated structure. Its radial correlation length is limited by the beam in every case, which excludes non-axisymmetric structure on radial scales larger than ~1.5 pressure scale heights. In three of the four disks the structure is instead resolved and elongated in azimuth, by factors of 1.6-2.3, as expected of small-scale perturbations sheared out by Keplerian rotation. We further introduce two statistics that characterize spiral structure without tracing an arm crest: a modal power spectrum, measuring the periodicity of the azimuthal structure, and a winding contrast, measuring the radial dependence of its phase. The velocity residuals of three disks are significantly wound. MWC 758 is the only source in which the winding is coherent across the disk and shared by the temperature field. Its trailing m=1 temperature structure is consistent with the illumination of a warp, while its velocity field carries small-scale, sheared structure that could redistribute angular momentum. Because neither statistic requires a crest, both extend a quantitative description of winding to the flocculent regime, where a spiral cannot be parameterized.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.00195-b31b1b.svg)](https://arxiv.org/abs/2611.00195) | **EMBER: A Little Red Dot Candidate at $z_{\rm spec}=10.36$ in the Seed Black Hole Era**  |
|| T. S. Tanaka, et al. -- incl., <mark>Y. Wu</mark>, <mark>Y. Zhang</mark> |
|*Appeared on*| *2026-11-03*|
|*Comments*| *17 pages, 5 figures, 2 tables*|
|**Abstract**|            Little red dots (LRDs) are a newly discovered population that may trace rapidly growing black holes (BHs) and provide a link between seed BHs and luminous active galactic nuclei (AGNs). However, no LRD had previously been spectroscopically confirmed at $z>10$, leaving this earliest phase observationally unexplored. Using JWST/NIRSpec spectroscopy from the EMBER program, we measure a spectroscopic redshift of $z_{\rm spec}=10.361^{+0.003}_{-0.002}$ for \target, the highest-redshift LRD candidate known to date. The redshift is constrained by a prominent Ly$\alpha$ break and multiple tentative emission lines, including C\,{\sc iv}\,$\lambda\lambda1549,1551$ and [O\,{\sc iii}]\,$\lambda4363$ detected at $S/N=3.9$ and $3.6$, respectively. Its extreme NIRCam-MIRI color of $m_{\rm F444W}-m_{\rm F770W}=1.82\pm0.1$, characteristic V-shaped continuum with a turnover near the Balmer limit, and continuum slopes of $\beta_{\rm UV}=-2.26^{+0.11}_{-0.11}$ and $\beta_{\rm opt}=1.17^{+0.74}_{-0.70}$ all support its LRD nature. A possible iron complex at $\lambda_{\rm rest}\sim0.24$--$0.25\,{\rm \mu m}$ and a possibly high $\log\left([{\rm O\,III]\,\lambda4363/\rm H\gamma}\right)=0.17^{+0.24}_{-0.22}$ further support an AGN-like interpretation, as seen in previously known LRDs. Tentative detection of Ly$\alpha$ emission may indicate enhanced Ly$\alpha$ transmission at $z\sim10$. Our findings extend the observational frontier of BH growth into the seed-BH formation era, and upcoming MIRI spectroscopy will further establish the physical nature of the source.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: '69117' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.00202-b31b1b.svg)](https://arxiv.org/abs/2611.00202) | **A Supermagnified Little Red Dot with a Resolved Host**  |
|| J. Judež, et al. -- incl., <mark>J. Matharu</mark> |
|*Appeared on*| *2026-11-03*|
|*Comments*| *Submitted to ApJ Letters. 12 pages including references. It contains 3 figures in the main text + 1 figure in the appendix and 2 tables in the main text*|
|**Abstract**|            Little red dots (LRDs) are an intriguing new class of sources in the early universe discovered by the James Webb Space Telescope (JWST) characterized by V-shaped spectra and extremely compact sizes. Their underlying mechanisms still remain the subject of ongoing debate. We present JWST observations of a gravitationally lensed LRD nicknamed "Best Friend" at $z_{\rm spec} = 8.297\pm0.002$. Its high magnification factor of $\mu = 36^{+8}_{-12}$ makes it the highest magnified LRD reported to date and unveils both the resolved structure of its host and a central engine less than 7 pc in size. The spectrum of the central engine, in which broad lines are not detected, possibly owing to the limited S/N and spectral resolution, is better interpreted as black hole activity rather than a dense star cluster. The latter cannot be fully excluded, but if true, the resulting lower limit on stellar density would be $>1.25^{+0.4}_{-0.3}\cdot10^6 \mathrm{M_{\odot}/pc}^2$, which is around 2 orders of magnitude higher than stellar densities of typical clusters at high-z. By performing a spatially resolved analysis of NIRCam photometry and NIRSpec/PRISM spectroscopy, we find that the host stellar mass is $\log(M_\star/M_{\odot}) = 7.3^{+0.2}_{-0.8}$. Best Friend demonstrates the power of lensing to reveal the morphologies of LRDs.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |

... (3 outputs left out)


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.02064-b31b1b.svg)](https://arxiv.org/abs/2611.02064) | **Effects of Competing Reactions on $^{59}\mathrm{Cu}$ and $^{63}\mathrm{Ga}$ on Abundance Distributions in Type I X-Ray Bursts**  |
|| Y. Guan, et al. -- incl., <mark>Y. Wang</mark> |
|*Appeared on*| *2026-11-03*|
|*Comments*| **|
|**Abstract**|            The composition of X-ray burst ashes affects the thermal transport properties of the neutron star crust, as well as heating and cooling through weak interaction processes. Reaction-rate uncertainties can produce secondary peaks separated from the main peak in final abundance distributions. Using STARLIB reaction rates and their temperature-dependent uncertainties in the K04 one-zone post-processing model, we combine four sets of Monte Carlo calculations with XGBoost classification, SHAP analysis, and reaction-flow analysis to investigate the origins of these secondary peaks. We find that the competing $(p,\alpha)$ and $(p,\gamma)$ reaction pairs on $^{59}\mathrm{Cu}$ and $^{63}\mathrm{Ga}$ are primarily responsible for secondary peaks outside the central 68\% confidence interval in all 44 isotopes examined. Fixing both pairs at their median rates reduces the fractions of samples in the secondary-peak regions by 96\%--100\%, while the bounds of the central 68\% interval remain largely unchanged for most mass numbers. The $^{59}\mathrm{Cu}$ pair has the larger effect. Enhanced recycling limits the supply of material to heavier isotopes and accelerates proton consumption, while earlier proton exhaustion further suppresses subsequent proton captures. These changes alter the balance between isotope production and destruction, leading to high- or low-abundance secondary peaks over a broad mass range beyond the Ni--Cu and Zn--Ga cycles. These results identify $^{59}\mathrm{Cu}(p,\gamma)^{60}\mathrm{Zn}$ and the two competing channels on $^{63}\mathrm{Ga}$ as priorities for further nuclear-physics constraints to improve XRB abundance predictions.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.02397-b31b1b.svg)](https://arxiv.org/abs/2611.02397) | **A Massive and Eccentric Binary Millisecond Pulsar in the Globular Cluster M10**  |
|| <mark>Y. Wang</mark>, et al. |
|*Appeared on*| *2026-11-03*|
|*Comments*| *13 pages, 4 figures*|
|**Abstract**|            We present the timing solution of the millisecond pulsar PSR J1657-0406A in the globular cluster M10, based on observations performed with the Five-hundred-meter Aperture Spherical radio Telescope (FAST). The timing results show that PSR J1657-0406A is in a long-period ($P_b \sim 37$ days) and eccentric ($e \sim 0.12$) orbit. We also measured the periastron advance $\dot{\omega} = 0.00090(13)$ degrees per year, which implies a total system mass of $2.5(6)$ solar masses, assuming that general relativity accurately models the orbital motion. No optical counterpart was detected within the positional uncertainty, indicating that the companion is unlikely to be a main-sequence star. Considering possible formation channels for the orbital eccentricity, we favor a helium white dwarf companion, although a massive white dwarf or neutron star cannot be completely excluded.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.02934-b31b1b.svg)](https://arxiv.org/abs/2611.02934) | **Revealing the dusty shell of the semi-regular variable V CVn**  |
|| Y. Nikolov, et al. -- incl., <mark>S. Kraus</mark> |
|*Appeared on*| *2026-11-03*|
|*Comments*| *Accepted for publication in Astronomy & Astrophysics on 21 October 2026*|
|**Abstract**|            The semi-regular variable (SRV) star V CVn exhibits a highly variable optical linear polarization, yet its position angle (PA) has remained remarkably stable at approximately 100 +/- 20 deg. over several decades. We present a comprehensive multi-technique study of the system covering six pulsation cycles from 2020 to 2025. This campaign includes high-cadence photopolarimetry, spectropolarimetry, and the first interferometric observations of V CVn using the CHARA Array (MIRC-X and MYSTIC) in 2024 and 2025. Our photopolarimetric data reveal a significant increase in polarization, reaching peaks of up to 6% during the 2023-2025 cycles, while the characteristic anticorrelation with brightness and the stability of the PA persisted. The interferometric observations resolve the stellar photosphere but show no evidence of large-scale asymmetric structures, such as a bow shock or isolated scattering arcs, at the scales and sensitivities probed. Based on these results, we exclude previous models based on interstellar medium (ISM) interactions or static reflection clouds as the primary drivers of the observed polarization. Instead, we propose a circumstellar disk or flattened envelope, analogous to the L2 Puppis system, as the most plausible explanation. We hypothesize that this stable geometry may be shaped or maintained by the star's high proper motion, where ram pressure from the ISM interaction favors a flattened distribution of circumstellar material. Within this stable framework, the polarization peaks are interpreted as the result of stochastic dust clumps, likely triggered by the star's long secondary period (LSP), which causes episodic increases in mass loss. This combination of an ISM-influenced disk geometry and transient, LSP-driven dust clumps naturally accounts for both the decadal stability of the position angle and the highly variable polarization amplitude of V CVn.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.00224-b31b1b.svg)](https://arxiv.org/abs/2611.00224) | **Euclid Quick Data Release (Q1). Mass-to-light ratio of elliptical galaxies: constraints from strong lensing**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-03*|
|*Comments*| *Submitted to Astronomy & Astrophysics*|
|**Abstract**|            Mass is among the most fundamental properties describing a galaxy. The relation between the luminosity and the total mass of a galaxy contains information on the efficiency of the star formation process and on the distribution of dark matter. We used strong gravitational lensing data from the Euclid Quick Data Release 1 (Q1) to constrain the total mass enclosed within an aperture of $5\,h^{-1}$kpc, $M_5$, of elliptical galaxies. Our analysis is based on a sample of 95 lenses, which we built by ensuring that their selection function is reproducible. This enabled us to correct our inference for selection effects. The average $\log_{10}{(M_5\,h\,M_\odot^{-1})}$ for a galaxy with an $r$-band luminosity of $10^{10.7}L_{\odot}\,h^{-2}$ at $z=0.5$ is $11.30\pm0.02$. The mass $M_5$ scales with luminosity to the power $\beta_\mathrm{L}=0.71\pm0.07$, and decreases with increasing redshift at fixed luminosity. Assuming a scenario in which galaxies grow by mergers and stellar populations evolve passively, these correlations imply that $M_5$ increases with time for individual galaxies. These measurements provide a useful benchmark for testing models of galaxy structure and evolution that make predictions on the distribution of total mass on the scales of $5\,h^{-1}$kpc.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.00215/./CompositePlotsStaga.png', 'tmp_2611.00215/./CompositePlotsKeynote163222a.png', 'tmp_2611.00215/./CompositePlotsK183400.png', 'tmp_2611.00215/./CompositePlots182413a.png', 'tmp_2611.00215/./CompositePlotsKWDCa.png', 'tmp_2611.00215/./CompositePlots182335a.png', 'tmp_2611.00215/./CompositePlotsHH510.png', 'tmp_2611.00215/./Ratio_RTsol_nD1e6_I01_l11000_D1pc_2.png', 'tmp_2611.00215/./allSpectra_mainPeak.png']
copying  tmp_2611.00215/./CompositePlotsStaga.png to _build/html/
copying  tmp_2611.00215/./CompositePlotsKeynote163222a.png to _build/html/
copying  tmp_2611.00215/./CompositePlotsK183400.png to _build/html/
copying  tmp_2611.00215/./CompositePlots182413a.png to _build/html/
copying  tmp_2611.00215/./CompositePlotsKWDCa.png to _build/html/
copying  tmp_2611.00215/./CompositePlots182335a.png to _build/html/
copying  tmp_2611.00215/./CompositePlotsHH510.png to _build/html/
copying  tmp_2611.00215/./Ratio_RTsol_nD1e6_I01_l11000_D1pc_2.png to _build/html/

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\HII}{\mbox{H {\footnotesize II}}}$
$\newcommand{\thebibliography}{\DeclareRobustCommand{\VAN}[3]{##3}\VANthebibliography}$</div>



<div id="title">

# A VLT/CRIRES+ study of helium 1.083$\mu$m absorption features towards the inner Orion Nebula 

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.00215-b31b1b.svg)](https://arxiv.org/abs/2611.00215)<mark>Appeared on: 2026-11-03</mark> -  _Accepted for publication in MNRAS_

</div>
<div id="authors">

T. J. Haworth, et al. -- incl., <mark>M. McCaughrean</mark>

</div>
<div id="abstract">

**Abstract:** JWST NIRCam F115W observations towards the inner Orion Nebula revealed a diverse array of previously unknown dark features. The most likely cause was absorption or suppressed emission of the helium 1.083 $\mu$ m triplet which lies in that filter. We have obtained VLT/CRIRES+ high-resolution spectrographic observations over a selection of different dark targets, confirming that the features are indeed explained by a reduction in the flux of the helium 1.083 $\mu$ m triplet, a reduction that is fairly uniform across the line. The Orion Nebula is a blister H ii region, with a dense, bright ionised layer on the far side from the observer near the Orion A molecular cloud and lower density gas elsewhere. Using a simple radiative transfer model, we show that dark features arise when emission from the dense layer is absorbed by intermediate density helium (e.g. in clumps, globules, or winds from objects like proplyds) on the near side of the dense layer.

</div>

<div id="div_fig1">

<img src="tmp_2611.00215/./CompositePlotsStaga.png" alt="Fig11.1" width="14%"/><img src="tmp_2611.00215/./CompositePlotsKeynote163222a.png" alt="Fig11.2" width="14%"/><img src="tmp_2611.00215/./CompositePlotsK183400.png" alt="Fig11.3" width="14%"/><img src="tmp_2611.00215/./CompositePlots182413a.png" alt="Fig11.4" width="14%"/><img src="tmp_2611.00215/./CompositePlotsKWDCa.png" alt="Fig11.5" width="14%"/><img src="tmp_2611.00215/./CompositePlots182335a.png" alt="Fig11.6" width="14%"/><img src="tmp_2611.00215/./CompositePlotsHH510.png" alt="Fig11.7" width="14%"/>

**Figure 11. -** Normalised CRIRES+ spectra integrated along the slit for one pointing towards each of our seven targets. The blue dotted line is the full spectrum, while the orange solid line is the CRIRES+ spectrum multiplied by the throughput of the F115W filter (green dashed line). The insets show an expanded view of the [S iii] and He i triplet lines. The inset image shows the target and slit orientation. The full set of integrated spectra along the slit for all pointings is shown in Appendix \ref{sec:allFullSpectra}.  (*fig:fullSpectrumAll*)

</div>
<div id="div_fig2">

<img src="tmp_2611.00215/./Ratio_RTsol_nD1e6_I01_l11000_D1pc_2.png" alt="Fig6" width="100%"/>

**Figure 6. -** The ratio of the emergent intensity from a LOS intersecting a 1000 au thick region with a density of $10^3$ cm$^{-3}$ to that from the ambient medium as a function of the density and thickness of the foreground layer. There are two regimes: one in which the foreground medium is optically thick and one in which the bright background is attenuated by the overdensity. The dashed line denotes the transition between the  two regimes.  (*fig:RTSolResults*)

</div>
<div id="div_fig3">

<img src="tmp_2611.00215/./allSpectra_mainPeak.png" alt="Fig2" width="100%"/>

**Figure 2. -** {The helium triplet line profile for all slits across our main targets, excluding HH 510. Each spectrum has been aligned such that the peak is at the $2^3S_1\rightarrow2^3P_2$ transition. The three vertical dashed lines show the vacuum rest frame wavelengths of the helium triplet. In all cases, there is a red shoulder due to back scattering. In 182-335, there is a hint of emission from the `near ionised layer' (NIL; see main text) offset by around 15--20 km s$^{-1}$ (see the 19 km s$^{-1}$ scale bar), but this is generally not the case. The smaller feature offset by about 0.2 nm ($2\times10^{-4} \mu$m) from the peak is possibly attributed to the CI $^1{\rm F}^0-1/2[7/2]$ transition emitted from the background PDR and is unlikely to be telluric as the offset from the main helium feature is independent of the barycentric correction.} (*fig:detailedLook*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.00215"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\arraystretch}{1.1}$</div>



<div id="title">

# exoALMA XXV: A Statistical Analysis of Gaseous Substructure in Low-Inclination Disks

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.01209-b31b1b.svg)](https://arxiv.org/abs/2611.01209)<mark>Appeared on: 2026-11-03</mark> -  _Accepted to exoALMA ApJL Focus Issue_

</div>
<div id="authors">

R. Teague, et al. -- incl., <mark>M. Benisty</mark>, <mark>M. Flock</mark>

</div>
<div id="abstract">

**Abstract:** Protoplanetary disks exhibit complex structure in both their gas dynamics and their temperature. Characterizing it has proven challenging, owing to the mismatch between the observed complexity and the relatively simple semi-analytical models typically used to interpret it. To augment these analyses, we apply the second-order structure function to the velocity and temperature perturbations of four low-inclination sources from the exoALMA program, each measured against a noise-only null built from the data themselves. All exhibit correlated structure. Its radial correlation length is limited by the beam in every case, which excludes non-axisymmetric structure on radial scales larger than ${\sim}1.5$ pressure scale heights. In three of the four disks the structure is instead resolved and elongated in azimuth, by factors of $1.6$ -- $2.3$ , as expected of small-scale perturbations sheared out by Keplerian rotation. We further introduce two statistics that characterize spiral structure without tracing an arm crest: a $*modal power*$ spectrum, measuring the periodicity of the azimuthal structure, and a $*winding contrast*$ , measuring the radial dependence of its phase. The velocity residuals of three disks are significantly wound. MWC 758 is the only source in which the winding is coherent across the disk and shared by the temperature field. Its trailing $m=1$ temperature structure is consistent with the illumination of a warp, while its velocity field carries small-scale, sheared structure that could redistribute angular momentum. Because neither statistic requires a crest, both extend a quantitative description of winding to the flocculent regime, where a spiral cannot be parameterized.

</div>

<div id="div_fig1">

<img src="tmp_2611.01209/figures/polar_residuals.png" alt="Fig5" width="100%"/>

**Figure 5. -** Velocity (top) and temperature (bottom) residuals for the four disks shown in a polar projection. Note that the bottom row is flipped upside down. The inner radius for all panels is $0$\farcs$3$, twice the beam major axis and the radial axis is shown in log-space to highlight logarithmic spiral structure. The velocity residuals for HD 143006 and J1604 have been inflated by a factor of two and three, respectively, to make them visible with this color scale. (*fig:polar_residuals*)

</div>
<div id="div_fig2">

<img src="tmp_2611.01209/figures/onsky_residuals.png" alt="Fig4" width="100%"/>

**Figure 4. -** The on-sky projection of the velocity residuals (top) and peak brightness temperature residuals (bottom) for the four sources. The black circle in the lower left shows the $0$\farcs$15$ CLEAN beam and the scale bar in the lower right shows 100 au. Note that for HD 143006 and J1604, the velocity residuals have been inflated by a factor of two and three, respectively, such that they are visible, while additionally HD 143006 has had its shadow removed in the peak brightness temperature residual. All maps are masked where the peak brightness temperature drops below 7 K. All four disks rotate clockwise on the sky bar HD 135344B, which rotates counter-clockwise \citep[][see Table \ref{tab:sources}]{Izquierdo_ea_2025}. (*fig:onsky_residuals*)

</div>
<div id="div_fig3">

<img src="tmp_2611.01209/figures/modal_power_radial.png" alt="Fig6" width="100%"/>

**Figure 6. -** Modal power as a function of radius for each source and both channels. The y-axis shows the fraction of the total azimuthal variance at a given radius that can be accounted for by $m = 1 - 4$ modes. Darker swathes represent lower-order terms. The top row shows the velocity residual channel, while the bottom shows the peak intensity channel. $m=1$ modes appear to be more prevalent in the intensity channel than the velocity as the $m=1$ modes were subtracted as part of the background model as described in Section \ref{sec:substructures:velocity}. (*fig:modal_power_radial*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.01209"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

131  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

8  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

2  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
